# Fine-tune do revisor (LoRA) — Code Sentinela

Treina o **Qwen2.5-Coder-1.5B-Instruct** para revisar diffs Java/Spring no formato do Code Sentinela.

**Como usar:** no Google Colab, `Ambiente de execução > Alterar o tipo > GPU T4` (grátis). Rode as células em ordem.
Na célula de upload, envie o `treino.jsonl` gerado pelo `gerar_dataset.py`.

> Este notebook não foi executado pelo autor do projeto ainda. As versões estavam corretas em 12/11/2026;
> se algo mudar de nome nas bibliotecas, ajuste conforme a documentação do Unsloth.

In [ ]:
!pip install -q unsloth==2026.11.4

In [ ]:
from google.colab import files
enviado = files.upload()  # escolha o treino.jsonl
assert "treino.jsonl" in enviado, "envie um arquivo chamado treino.jsonl"

In [ ]:
from unsloth import FastLanguageModel

# Modelo base em 4 bits: cabe na T4 grátis (16 GB) com folga.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

# LoRA: em vez de mexer nos 1,5 bilhão de pesos, treina matrizes pequenas "ao lado" deles.
# r = tamanho dessas matrizes. 16 é um meio-termo comum entre capacidade e memória.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from datasets import load_dataset

dados = load_dataset("json", data_files="treino.jsonl", split="train")

def para_texto(exemplo):
    # Aplica o mesmo "chat template" que o modelo usa: sistema, usuário (diff) e assistente (JSON esperado).
    return {"text": tokenizer.apply_chat_template(exemplo["messages"], tokenize=False)}

dados = dados.map(para_texto)
print(dados[0]["text"][:800])

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

treinador = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dados,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=2048,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # lote efetivo = 8 exemplos
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=5,
        optim="adamw_8bit",
        seed=42,
        output_dir="saida",
        report_to="none",
    ),
)
# Só a resposta do assistente conta para o erro: o modelo aprende a RESPONDER, não a repetir o diff.
treinador = train_on_responses_only(
    treinador,
    instruction_part="<|im_start|>user
",
    response_part="<|im_start|>assistant
",
)
estatisticas = treinador.train()
print(estatisticas)

In [ ]:
# Teste rápido com um exemplo do próprio dataset
FastLanguageModel.for_inference(model)
mensagens = dados[0]["messages"][:2]  # sistema + usuário, sem a resposta
entrada = tokenizer.apply_chat_template(mensagens, add_generation_prompt=True, return_tensors="pt").to("cuda")
saida = model.generate(input_ids=entrada, max_new_tokens=200, temperature=0.0, do_sample=False)
print(tokenizer.decode(saida[0][entrada.shape[1]:], skip_special_tokens=True))
print("ESPERADO:", dados[0]["messages"][2]["content"])

In [ ]:
# Exporta em GGUF (formato do Ollama), quantizado em 4 bits: ~1 GB
model.save_pretrained_gguf("sentinela-lora", tokenizer, quantization_method="q4_k_m")
import glob
gguf = glob.glob("sentinela-lora/*.gguf") + glob.glob("*.gguf")
print(gguf)
files.download(gguf[0])

## Depois do download

No seu PC, na pasta `finetune/`, coloque o `.gguf` baixado ao lado do `Modelfile` e rode:

```
ollama create sentinela-lora -f Modelfile
uv run python avaliar.py --modelo ollama:sentinela-lora --teste dados/teste.jsonl
```

Compare com `--modelo ollama:qwen2.5-coder:1.5b` (o mesmo modelo sem treino) e com `--modelo claude`.